<!--- Made by:
      Oscar Antonio Restrepo Gutiérrez
--->

# Numerical stability and error propagation
[Local and global error.](#Error_a_nivel_local)<br>
[Algorithmic stability.](#Estabilidad_algoritmica)<br>
[Linearity and non-linearity in computation time.](#Linealidad_y_no_linealidad)<br>
[Supplementary material.](#Material_complementario)<br>


## Local and global error
<a id='Error_a_nivel_local'></a>
Regardless of the error's origin (see [types of errors](05_Error_Theory.ipynb#Tipos_de_errores)), in computing the error can be classified as local and global; local error refers to the error at each step of the iteration and arises from errors in additions/subtractions and multiplications/divisions, while global error refers to the accumulation of errors as operations and iterations are carried out. Here we'll see how they are computed and how they affect the stability of algorithms. But first let's define what is meant by error:

Let $x$ be any quantity and $x^*$ an approximation; the *absolute error* is defined as $\epsilon=|x^*-x|$ and the *relative error* as $\eta=\epsilon/|x|$, so that $x^*=x\pm\epsilon$, where the $\pm$ sign means the quantity is bounded to the interval $[x-\epsilon,x+\epsilon]$.


### Error in additions and subtractions
From error theory we know that if $x^*=x \pm \epsilon_x$ and $y^*=y \pm \epsilon_y$, the error in the sum is given by,

$$x^*+y^*=(x + y) \pm(\epsilon_x + \epsilon_y)$$

In subtraction the error is more complicated (see [subtractive cancellation](05_Error_Theory.ipynb#Error_de_cancelación_sustractiva)), since if the figures are very similar there is loss of significance. To see this more clearly, from the definition of relative error we know the error is proportional to the true quantity, i.e. $\epsilon_x=x\eta_x$, $\epsilon_y=y\eta_y,$ such that $x^*=x(1\pm\eta_x)$, $y^*=y(1\pm\eta_y)$; then the error is,

$$
\begin{align}
x^*-y^*=\,&x(1\pm\eta_x)-y(1\pm\eta_y)\\
   =\,&(x-y)\pm(x\eta_x-y\eta_y)\\
   =\,&(x-y)\left(1\pm\frac{|x\eta_x-y\eta_y|}{|x-y|}\right)\\
   =\,&(x-y)\left(1\pm\eta_{xy}\right)
\end{align}
$$

the relative error in this case will be,

$$
\epsilon_{rel}=\eta_{xy}=\frac{|x\eta_x-y\eta_y|}{|x-y|},
$$

which tells us that if $x$ is very similar to $y$, then small values of $\eta_x$ and $\eta_y$ can produce large errors in the final result.

### Example: error accumulation in addition and subtraction
The sequence $x_{n+1}=(x_n-1)10$ with $x_0=10/9$ should, symbolically, generate the same value $x_n=10/9$ at every step for $n=1,2,...$ (check this with pencil and paper); nevertheless, the following numerical code shows the loss of precision at each iterative step depending on the number of significant figures taken to approximate $10/9\approx 1.111111...$ — for example, if you take only 10 significant digits, one digit is lost at each iteration. Cases like this are not easy to detect; let's compare what happens if we use 5, 7, 10 or 16 digits after the point:  


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
# *** Gradual loss of significance *** 

# Change this variable and see what happens (compare results):
digits = 10 #5, 7, 8, 10, 16 # number of digits after the point.
x=np.round(10.0/9.0,digits) # x = 1.111111111 ...
for i in range(30):
    print(i,x)
    x=(x-1.0)*10.0


### Local error in multiplication and division
In this case, if $x^*=x + \epsilon_x$ and $y^*=y + \epsilon_y$, error theory gives the error as,

$$x^*y^*=xy \pm (x\epsilon_y+y\epsilon_x),$$

again, in this case it's better to consider $x^*=x(1\pm\eta_x)$ and $y^*=y(1\pm\eta_y)$, which gives,

$$
\begin{align}
x^*y^*=\,&x(1\pm\eta_x)y(1\pm\eta_y)\\
      =\,&xy(1\pm\eta_x\pm\eta_y\pm\eta_x\eta_y)\\
      \approx\,&xy[1\pm(\eta_x+\eta_y)]
\end{align}    
$$

where $\eta_x\eta_y$ is assumed to be very small and is neglected. The error in division is similar, since $x/y$ can be seen as the product $x\times(1/y)$, i.e.,
$$
\begin{align}
\frac{x^*}{y^*}=\,&\frac{x(1\pm\eta_x)}{y(1\pm\eta_y)}\\
               =\,&\frac{x}{y}(1\pm\eta_x)(1\pm\eta_y)^{-1}\\
         \approx\,&\frac{x}{y}(1\pm\eta_x)(1\mp\eta_y)\\
         \approx\,&\frac{x}{y}(1\pm\eta_x\mp\eta_y\pm\eta_x\eta_y)\\
         \approx\,&\frac{x}{y}[1\pm(\eta_x-\eta_y)].
\end{align}    
$$
As can be seen, the error appears smaller than in multiplication; moreover, if $\eta_x$ and $\eta_y$ are equal, the error would be of order $\eta_x\eta_y$, the neglected term. Nevertheless, it's better to consider the errors in absolute value, i.e. the error is the same as in multiplication.

### The error for a function
According to error theory it is given by,
$$f^*=f\pm\Delta_f,$$
where,
$$\Delta_f=|f'(x)|\epsilon_x,$$

note that this is the change in the function between $x$ and $x^*$ (to understand this, think of derivatives and of $\Delta_f$ as the differential of the function).
So for example, consider $f(x)=x^2$; then the error in the function is,

$$\Delta_f=|2x|\epsilon_x.$$

Let's see this explicitly,

$$
\begin{align}
f(x^*)=\,&(x\pm\epsilon_x)^2\\
        =\,& x^2\pm 2x\epsilon_x+\epsilon_x^2\\
        \approx\,& x^2\pm 2x\epsilon_x\\
        \approx\,&f(x)\pm\Delta_f,
\end{align} 
$$

where the error matches what was expressed previously; note that $\epsilon_x^2$ is neglected since it is very small.



## Algorithmic stability
<a id='Estabilidad_algoritmica'></a>

Characterising an algorithm by its stability refers to whether the error grows slowly or quickly as iterations proceed; this is known as the stability of the initial conditions and is directly related to computing the global error. To understand this better, let's apply the definition of error to a function $f(x)$; let $x$ be the input data and $\epsilon$ the error in the input data. Then we say the algorithm is *numerically stable* if the absolute error is,

$$x - (x + \epsilon) = \epsilon  \propto |f(x) - f(x + \epsilon)|$$

with relative error,

$$\frac{x - (x + \epsilon)}{x}  \propto \left|\frac{f(x) - f(x + \epsilon)}{f(x)}\right|.$$

The algorithm is *numerically unstable* if the errors satisfy,

$$x - (x + \epsilon) = \epsilon << |f(x) - f(x + \epsilon)|$$

$$\frac{x - (x + \epsilon)}{x} << \left|\frac{f(x) - f(x + \epsilon)}{f(x)}\right|$$

There are two important types of algorithms according to their stability, *linear* and *exponential*; let's look at how the error propagates with iterations:


### Algorithm with linear error
Suppose an error $\epsilon_0$ is introduced at some point in the calculations; then if the error after $n$ steps is $\epsilon_n \approx Cn\epsilon_0$, the error is linear; if $\epsilon_n \approx C^n\epsilon_0$ then the error is exponential. For example,
consider the algorithm given by the iteration:

$$p(n) = 2p(n−1) − p(n−2),    n=2,3,⋯$$

which has the solution,

$$p(n)= A + Bn,$$

since,

$$
\begin{eqnarray}
2p(n−1) − p(n−2) &=& 2(A + B(n − 1)) − (A + B(n − 2))\\
&=& A(2 − 1) + B(2n − 2 − n + 2)\\
&=& A + Bn\\
&=& p(n).
\end{eqnarray}
$$

If we choose $p(0) = 1$ and $p(1) = 1/3$, then solving gives $A = 1$ and $B = −2/3$.
Now, if we approximate $\hat p(1)=0.33333$, this gives $B = -0.66667$ (with $5$ significant figures), so that,

$$ p(n) = 1 − \frac{2}{3}n,$$

and

$$\hat p(n) = 1.0000 − 0.66667n,$$

so the error grows linearly with $n$,

$$\epsilon_n=|p(n) − \hat p(n)| = \left(0.66667 − \frac{2}{3}\right)n.$$

Let's show that the error grows linearly with the iterations,


In [ ]:
# Number of iterations
Niter = 10000

# 16-digit approximation (64 bits)
A_d = 1.0
B_d = -2/3.
# 7-digit approximation (32 bits)
A_s =  1.00000
B_s = -0.66667

# function p(n): solution to the n-th term
pn = lambda A, B, n: A + B*n

# Arrays to store the iterations
p_d = []
p_s = []
narray = range(Niter)
for n in narray:
    p_d.append( pn( A_d, B_d, n ) )
    p_s.append( pn( A_s, B_s, n ) )

# Convert to numpy arrays
p_d = np.array(p_d)
p_s = np.array(p_s)

error = (p_d - p_s)# /p_d
plt.plot( narray, error, "-", color="blue" )
plt.xlabel("Number of iterations $n$", fontsize=14)
plt.ylabel(r"Error $p_n-\hat{p}_n$", fontsize=14)
plt.grid(True)
plt.show()


Now let's compare the error propagation against the exact solution using the iteration formula:


In [ ]:
# %pylab qt
N_iter = 10000 # Number of iterations

# 64-bit precision
A_d =  1
B_d = -2/3

#----------------------------------
pd = np.zeros(N_iter)
ps = np.zeros(N_iter)
pe = np.zeros(N_iter) 

pd[0]=pe[0]=1.0;  ps[0] = 1
pd[1]=pe[1]=1/3;  ps[1] = 0.33333

for n in range(2,N_iter):
    pd[n] = 2*pd[n-1] - pd[n-2]
    ps[n] = 2*ps[n-1] - ps[n-2]
    pe[n] = A_d + B_d*n
#----------------------------------

error_d = abs(pd - pe) # double, 64 bits
error_s = abs(ps - pe) # 5 significant figures
narray  = np.arange(N_iter)

plt.plot( narray, error_d, "-", color="blue",label='exact')
plt.plot( narray, error_s, "--", color="red",label='approx' )
plt.legend()
plt.xlabel("Number of iterations $n$", fontsize=14)
plt.ylabel(r"Error $p_n-\hat{p}_n$", fontsize=14)

plt.grid(True) 
plt.show()



Comparing this plot to the previous one, we see they're identical. It's worth noting that "exact" here refers to using a 64-bit approximation, i.e. about 16 significant figures: $-2/3\approx -0.6666666666666667$.


### Algorithm with exponential error
Consider the algorithm produced by the following iteration:

$$p(n)=\frac{10}{3}p(n−1)−p(n−2), \quad   n=2,3,⋯$$

in this case it can be shown that the exact solution at step $n$ is given by,

$$p(n)=A\left(\frac{1}{3}\right)^n + B3^n,$$

for any $A$ and $B$, which can be computed from the initial conditions; so if we choose $p(0)=1$ and $p(1)=1/3$ then $A=1$ and $B=0$, so,

$$ p(n) = 1\left(\frac{1}{3}\right)^n,$$

but if we approximate $1/3$ by $0.33333$, this gives $\hat A = 1.0000$ and $\hat B = −0.12500 \times 10^{−5}$, so,

$$ \hat p(n) = 1.0000 \left(\frac{1}{3}\right)^n − 0.12500 \times 10^{−5}(3)^n,$$

and the error is,

$$\epsilon_n=|p(n) − \hat p(n)|= 0.12500 \times 10^{−5}(3^n),$$

which shows exponential growth with respect to $n$. Now let's see how the error grows exponentially with the iterations:


In [ ]:
%matplotlib inline
# Number of iterations
N = 100

A_d = 1.0
B_d = 0.

# function p(n): solution to the n-th term
pn = lambda A, B, n: A*(1.0/3.0)**n + B*(3.0)**n

# Arrays to store the iterations
p_s = [1.000000,0.333333]
p_d = [1.,1/3.]

narray = range(N)
for n in narray[2:]:
    p_s.append( 10/3.*p_s[n-1]-p_s[n-2] )
    p_d.append( pn( A_d, B_d, n ) )

# Convert to numpy array
p_d = np.array(p_d)
p_s = np.array(p_s)

# a more compact but harder-to-read alternative
#p_d = np.append([1.000000,0.333333], np.array([A_d*(3.0)**-n + B_d*(3.0)**n for n in narray[2:]]))
#p_s = np.append([1.,1/3.]          , np.array([10/3.*p_s[n-1]-p_s[n-2]      for n in narray[2:]]))

error = p_d - p_s
plt.semilogy( narray, error, "-", color="blue" )
plt.xlabel("Number of iterations $n$", fontsize=14)
plt.ylabel(r"Error $p_n-\hat{p}_n$", fontsize=14)
plt.grid(True)
plt.show()


Now let's compare how the error evolves when using the iterated formula versus the exact solution at each step $n$:


In [ ]:
# Propagation of exponential error for the equation:
#   p(n)=10/3p(n−1)−p(n−2),    n=2,3,⋯
# With analytical solution:
#   p(n) = A(1/3)^n + B3^n (A = 1 and B = 0, for p(0) =1 and p(1) = 1/3)

M = 40

# ----- Iterative solution: --------
SI = []
p0=1; p1=1/3
for n in range(2,M+1):
    p2 = 10/3.*p1 - p0 # iterative formula
    p0 = p1
    p1 = p2
    SI.append(p2) 

# ------ Analytical solution ------------
SE = [(1/3)**n for n in range(2,M+1)]    
n  = np.arange(2,M+1)

plt.plot(n,SI, color="red", linewidth=2, label="Iterated solution" )
plt.plot(n,SE, color="blue", linewidth=2, label="Exact solution" )
plt.legend()
plt.show()


As we can see the problem still persists, since $1/3 \approx 0.3333333333333333$ at 64 bits, i.e. it is still an approximation.


## Global error from the accumulation of random errors
In many cases the total global error after $N$ steps is not systematic and is random in origin; in this case it can be shown, by analogy with Brownian motion (we'll see this later at the end of the course), that the relative error is,

$$\epsilon_{re}=\sqrt{N}\epsilon_m$$
 
where $\epsilon_m$ is the machine error. So, for example, we can state that the total error in a convergence process will be,

$$\epsilon_{total}=\epsilon_{tr}+\sqrt{N}\epsilon_m,$$

where $\epsilon_{tr}$ is the truncation error after $N$ steps.


In [ ]:
# example/exercise


## Linearity and non-linearity in computation time
<a id='Linealidad_y_no_linealidad'></a>
An algorithm is also said to be linear with respect to computation time.
The following routine is linear, since the number of steps is proportional to the time. In this example we evaluate the time needed to compute the sum of $N$ random numbers as a function of how many numbers are added:
the sum is computed two ways — first via a `for` loop, and second using `numpy`:


In [ ]:
# uncomment and study the linearity in the plot by zooming in.
# %pylab
import time as tm                # library for timing 

nmax = 10**np.arange(1,7,0.1)    # Maximum number of iterations

t_for = []                       # Store python for loop times
t_np = []                        # Store numpy Python times
for n in nmax:
    N = np.random.random(int(n)) # Generate an array of n random elements 

    # **** Sum with for ****
    ti = tm.perf_counter()       # Measure start time
    suma = 0
    for i in range(int(n)):
        suma += N[i]
    tf = tm.perf_counter()       # Measure end time
    t_for.append(tf-ti)          # Store computation time
    
    # **** Sum with numpy ****
    ti = tm.perf_counter()       # Measure start time
    suma = np.sum(N)
    tf = tm.perf_counter()       # Measure end time
    t_np.append(tf-ti)           # Store computation time

# try plt.plot to see the linearity    
plt.semilogx( nmax, t_for, "-", color="red", linewidth=2, label="PYTHON SUM" )
plt.semilogx( nmax, t_np, "-", color="blue", linewidth=2, label="NUMPY SUM" )
plt.xlabel("$N$", fontsize=14)
plt.ylabel("computation time (seconds)", fontsize=14)
plt.legend()
plt.grid(True)
plt.show()


In the plot above, are the times linear or non-linear with respect to N? Yes, they're linear (note the plot is semi-logarithmic in $x$; to see this, re-plot with `plt.plot`).

In some problems the time is not linear with respect to the number of iterations; for example, in the N-body problem with gravitational interaction, computing the force takes a time of $N^2$ (see [exercise](#computo_N_cuerpos)), since the interaction between every pair of particles must be computed, i.e. two `for` loops are needed for the calculation — for particle one the distances $r_{1,2}, r_{1,3},...,r_{1,N}$ are computed, then this is repeated for particle two, and so on until finished; this is the second loop.
It can be shown that using other algorithms the time can be reduced to $N\log N$; let's see the difference between these two timings:


In [ ]:
N = np.arange(1,1e3,0.1)
plt.loglog( N, N**2, lw=2, label="$N^2$" ) # lw=linewidth is the line width.
plt.loglog( N, N*np.log(N), lw=2, label=r"$N\ \log N$" )
plt.legend( loc="upper left" )
plt.grid(True)
plt.show()


## Supplementary material
<a id='Material_complementario'></a>
In the first example we used $p(n)= A + Bn$ as the solution to the series generated by $p(n) = 2p(n−1)−p(n−2), n=2, 3$ (which defines a recursive sequence, i.e. each term of the sequence is defined as a function of previous terms); let's implement this series using the latter equation:


In [ ]:
# Solve the following equation
# p(n) = 2p(n−1)−p(n−2), n=2, 3,..., 10  with p(0)=1, p(1)=1/3
# a) recursively: 
def p(n):
    if n == 0: return 1
    if n == 1: return 1/3
    P = 2*p(n-1) - p(n-2)
    #print (n,P)
    return P

print ('Recursive way',p(10)) # gives -5.666666666666664


# Iteratively:
p0=1; p1=1/3
for n in range(2,11):
    p2 = 2*p1 - p0
    p0 = p1
    p1 = p2
    #print (n,P)
    
print ('Iterative way',p2) 


## Exercises
**Exercise**: Consider the function $f(x) =x\left(\sqrt{x+ 1}−\sqrt{x}\right)$, which for a sequence of powers of 10 gives the results:
```c
      x   f(x) computed   f(x) true
      1     0.414210         0.414214
     10     1.54340          1.54347
    100     4.99000          4.98756
   1000    15.8000          15.8074
 10,000    50.0000          49.9988
100,000   100.000          158.113
```
explain numerically the results and the significant loss of digits (hint: you must analyse term by term).

**Exercise**: Consider the function $(1 - \cos x)/\sin\,x$ with $x \approx 0$, i.e. consider the operation,
```python
x=1e-2
(1-np.round(cos(x),4))/np.round(sin(x),4) # gives 0.0
```
the answer should be approximately 0.005. Propose an alternative method to fix this problem.


**Exercise**: Consider the functions,

$$f(x)=\frac{1-\cos^2(x)}{x^2},\\
  g(x)= \frac{\sin^2\,(x)}{x^2},
$$

which are mathematically equivalent by the trigonometric identity $\sin^2\,(x)+\cos^2(x)=1$.

a) By the analytical result $f(0)=g(0)=1$, but if `x = 1e-5` is used the 32-bit numerical result is:
```python
x=np.float32(1e-5)
f(x),g(x) # gives (0.0, 1.0)
```
explain why this error occurs with $f(x)$ and not with $g(x)$ at 32 bits — what happens at 64 bits? (hint: compute the value of $\cos(x)$ and then its square, and compare — what effect does the minus sign have?).

b) Plot both functions on the interval $[-1,1]$ with 100 values, first at 32-bit precision and then at 64-bit precision,
```python
x=np.linspace(-0.1,0.1,100, dtype=float32) 
```
show that the results are catastrophic for the second plot. Explain why this result occurs.


**Exercise**: Euler's number is defined by the limit,
$e=\lim_{n\rightarrow\infty}(1 + 1/n)^n$,
use this formula to approximate $e$ for values of $n=1,...,20.$ Plot the error and explain why this formula fails as $n$ grows — is the error exponential or linear?

**Exercise**: the Fibonacci series has important applications in physics — for example, domains in superconductors are governed by Fibonacci numbers; in biology it describes the number of petals of daisies, how the density of branches on a tree trunk increases, and how the scales of a pine cone are arranged; and in astronomy it is used to describe pulsar stars, spiral galaxies and the formation of black holes. The iterative form of the Fibonacci sequence is defined by,

$$f_{n}=f_{{n-1}}+f_{{n-2}},$$

with initial condition $f_{0}=0$ and $f_{1}=1$.

a) Write a program that computes and plots the sequence up to $n=100$.

b) The Fibonacci sequence has the functional form,

$$
f_{n}=\frac{1}{\sqrt{5}}\left[\left({\frac {1+{\sqrt 5}}2}\right)^{n}-\left({\frac {1-{\sqrt 5}}2}\right)^{n}\right].
$$

Compute the error between this function and the iterated form. Which result is more accurate? Explain why.

c) The *golden ratio* is defined by the sequence $x_n=f_{n+1}/f_n$ and converges to $(1+\sqrt{5})/2$; write a program and compute this number to a desired precision.

e) The [Fibonacci spiral](https://en.wikipedia.org/wiki/Fibonacci_sequence#/media/File:Fibonacci_spiral_34.svg) is generated by drawing circular arcs connecting the opposite corners of squares fitted to the sequence's values; joining together squares of side 0, 1, 1, 2, 3, 5, 8, 13, 21 and 34, plot this spiral.

**Exercise**: [Spherical Bessel functions](https://en.wikipedia.org/wiki/Bessel_function#Spherical_Bessel_functions) $j_l(x)$ and $y_l(x)$ (the latter known as the spherical Neumann function and related to the former by $y_{l}(x)=(-1)^{l+1}j_{-1-l}(x)$) appear in many physics problems, such as electromagnetic waves, optics, heat conduction in cylindrical objects, vibration modes of a thin circular membrane, quantum mechanics, etc; they can be computed from the recurrence relations,

$$
j_{l+1}(x)=\frac{2l+1}{x}j_{l}(x)-j_{l-1}(x), \quad\hbox{(up),}\\
j_{l-1}(x)=\frac{2l+1}{x}j_{l}(x)-j_{l+1}(x), \quad\hbox{(down),}
$$

where the first two terms are,

$$
j_{0}(x)={\frac {\sin x}{x}},\quad  j_{1}(x)={\frac {\sin x}{x^{2}}}-{\frac {\cos x}{x}},\\
y_{0}(x)=-j_{-1}(x)=-\,{\frac {\cos x}{x}},\quad y_{1}(x)=j_{-2}(x)=-\,{\frac {\cos x}{x^{2}}}-{\frac {\sin x}{x}}.
$$

a) Write a program that shows that (up) produces a catastrophic error that grows as $N!$ (number of steps in the `for` loop), but that this error is corrected if (down) is used; compute up to $l=25$ for $x = 0.1, 1.0, 10.$ <br>
b) Plot the convergence and stability of the results (use a relative error of 1.0e-10). Hint, use:
```python
def J_down(x,n,m) : # Downward method 
    j=np.zeros(start+2)
    j[m+1] = j[m] = 1 
    for k in range(m,0,-1):
        j[k-1] = (2.*k+1.)/x*j[k]-j[k+1]
        scale = (np.sin(x)/x)/j[0] # scale the solution to j[0]
    return j[n]*scale
```
See Landau, section 2.2.

<a id='computo_N_cuerpos'></a>
**Exercise**: Consider a system with $N=2,...,20$ planets interacting via the law of universal gravitation; write a program that computes the interaction energies between each pair of planets. a) Verify that the computation time is proportional to $t\sim N^2$. b) Show that by restructuring the loops the computation time can be reduced to about half, $t\sim (N^2-N)/2$, by taking into account the symmetry $ r_{ij}=r_{ji}$ (hint: write out the matrix of $r_{ij}$ explicitly to see the symmetries). c) Write the code implementing part b).


## Bibliography
Burden

https://en.wikipedia.org/wiki/Numerical_stability

https://www.cl.cam.ac.uk/teaching/1819/NumAnalys/Numerical_Analysis_2019.pdf

http://www.math.pitt.edu/~trenchea/math1070/MATH1070_2_Error_and_Computer_Arithmetic.pdf
